# 第 3 章练习：PyTorch 与模型框架
这份练习要建立的不是“会调用导出 API”，而是完整的工件思维：模型状态怎样保存、计算图怎样导出、另一个 Runtime 怎样验证。

关闭完整实验后再做。每个任务先理解工件在软件栈中的角色，再补全 TODO。

## 准备模型与固定输入

**为什么运行这段代码？** 我们需要一个足够小、但包含参数和非线性算子的模型，用同一份输入贯穿 PyTorch、ONNX 和 ONNX Runtime。固定随机种子让权重和样本可复现。

`ExerciseMLP` 接收 `[Batch, 6]`，输出 `[Batch, 4]`。当前样本 Batch 为 5，但导出后 Batch 应允许变化。请先区分模型定义、模型参数和一次具体输入。

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
import onnx
import onnxruntime as ort
torch.manual_seed(7); artifacts=Path('artifacts'); artifacts.mkdir(exist_ok=True)
class ExerciseMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=6, out_features=12),
            nn.ReLU(),
            nn.Linear(in_features=12, out_features=4),
        )

    def forward(self, inputs):
        return self.net(inputs)

model = ExerciseMLP().eval()
sample = torch.randn(5, 6)
print('sample shape =', sample.shape)

# 任务一：保存与恢复权重

### 为什么做这个任务
`state_dict` 是参数与 Buffer 的映射，不包含 `ExerciseMLP` 的 Python 定义。恢复时必须先重建相同结构，再加载状态。这帮助你真正区分“模型结构”和“训练得到的状态”。

### 你需要完成什么
1. 用 `torch.save(model.state_dict(), checkpoint_path)` 保存状态；
2. 新建一个 `ExerciseMLP`；
3. 用 `torch.load(..., weights_only=True)` 读取状态；
4. 调用 `load_state_dict()` 并切换到 `eval()`；
5. 用相同输入比较原模型与恢复模型的输出。

**运行后观察什么？** 两个输出应该一致。若结构改变或漏载参数，加载过程应明确报错，而不是静默产生另一个模型。

In [ ]:
checkpoint_path=artifacts/'exercise_model.pth'
# TODO：保存 state_dict，重建模型并加载


# 任务二：导出 ONNX

### 为什么做这个任务
Checkpoint 面向 PyTorch 状态恢复；ONNX 则表达供其他 Runtime 读取的计算图。导出不是修改文件后缀，而是把框架运算映射成某个 Opset 下的 ONNX Operator。

### 你需要完成什么
1. 调用 `torch.onnx.export` 导出 `model` 和 `sample`；
2. 输入名设为 `input`，输出名设为 `output`；
3. 为输入和输出的第 0 维设置同名动态轴 `batch`；
4. 使用 Opset 17；
5. 用 `onnx.load` 和 `onnx.checker.check_model` 检查图。

**运行后观察什么？** 文件存在不代表图有效；Checker 通过说明结构满足 ONNX 规范，但仍不证明目标硬件支持全部算子。

<details><summary>需要提示时点这里</summary>`dynamic_axes` 同时为 `input` 和 `output` 指定 `{0: 'batch'}`，文件路径可以直接使用 `onnx_path`。</details>

In [ ]:
onnx_path=artifacts/'exercise_model.onnx'
# TODO：导出并使用 onnx.checker 检查


# 任务三：跨 Runtime 数值验证

### 为什么做这个任务
图能成功导出和加载，只说明格式正确；算子转换、精度或实现差异仍可能改变输出。因此必须使用同一输入比较源框架和目标 Runtime，不能只看程序是否报错。

### 你需要完成什么
1. 在 `no_grad()` 下执行 PyTorch 模型并转成 NumPy，赋给 `pytorch_output`；
2. 创建只使用 `CPUExecutionProvider` 的 `InferenceSession`；
3. 把 `sample.numpy()` 以输入名 `input` 传入 ORT；
4. 取输出名 `output` 的结果，赋给 `ort_output`；
5. 自己计算最大绝对误差并判断是否在 `1e-5` 内。

**运行后观察什么？** 两边 Shape 都应为 `[5, 4]`。很小的浮点误差通常可以接受，完全不比较数值则不能形成部署正确性的证据。

In [ ]:
# TODO：分别赋值 pytorch_output 和 ort_output
pytorch_output=None
ort_output=None

# 统一验收

验收器检查三类不同证据：Checkpoint 是否真实存在、ONNX 是否是有效计算图、PyTorch 与 ORT 输出是否在容差内。三项全部通过后生成 `artifacts/exercise_result.json`。

生成文件不是终点。只有工件能够被解析、执行，并产生可比较的结果，才算完成这条部署链路。

In [ ]:
from validate import validate_exercise,print_report,save_report
exercise_report=validate_exercise(globals()); print_report(exercise_report)
if exercise_report['passed']: save_report(exercise_report,artifacts/'exercise_result.json')